# SNR sweep with a moving target: the white-input sweep of notebook 05

Ignacio's notebooks 05 and 07 sweep the SNR at a fixed $-20$ dB target, and both print a caveat: as the
parameter grows every filter tends to the same NLMS floor, about $-\mathrm{SNR}$ dB, so a fixed target sits
closer to where the filters coincide the higher the SNR, and that alone pushes the ratio toward 1. The
reference for the SNR question is the moving target $-(\mathrm{SNR} + 15)$ dB of
`04_escenario_realista_large_k_mad.ipynb` (issue #8).

**What is already covered.** Notebook 07's sweep is the scenario of `04_escenario_realista_large_k_mad.ipynb`:

| | notebook 07 (Ignacio) | `04_escenario_realista_large_k_mad.ipynb` |
|---|---|---|
| input, response, noise | AR($-0.9$), room of section 7.1, GG $\beta^* = 0.2$ | the same |
| filters | sKF-L minorized, sKF-L joint | sKF-L minorized, sKF-L exact (marginal; the same filter as the joint, `vkf-kf.ipynb`) |
| $b_\eta$ | $\mathrm{E}\lvert\eta\rvert$ and $\sqrt{v_\eta/2}$ | $\mathrm{E}\lvert\eta\rvert$ (the original notebook 04: $\sqrt{v_\eta/2}$) |
| SNR, target | 0-20 dB, $-20$ dB | 0-30 dB, $-(\mathrm{SNR} + 15)$ dB |

So notebook 07 needs no new run: with the moving target and $\mathrm{E}|\eta|$ its answer is already on
record, exact / minorized 1.11, 1.09, 1.06, 1.01, 0.99, 0.96, 0.93 at 0 to 30 dB (DECISIONS.md, 2026-09-21).

**What is not.** Notebook 05's sweep: **white input**, and both families, sKF-L and fKF-L. This notebook
reruns it with the moving target: the same filters, grids, signals and search protocol as notebook 05
(branch `ignacio/joint-vs-marginal-vs-minorized`, commit `4e3102a`); only the target moves. The SNRs are
extended from notebook 05's 0-15 dB to 0-30 dB, to match notebook 04's row.

## 1. Imports

In [ ]:
import time

import numpy as np
from matplotlib import pyplot as plt
from scipy.signal import lfilter
from scipy.special import gammaln, log_ndtr
from scipy.stats import gennorm
import rir_generator as rir

%config InlineBackend.figure_format = 'svg'
NOTEBOOK_START = time.time()
print("imports ready")

## 2. The scenario

Notebook 05's: white input, room response of section 7.1, GG noise at $\beta^* = 0.2$, $b_\eta = \mathrm{E}|\eta_t|$.

In [ ]:
# Values of notebook 05 (branch ignacio/joint-vs-marginal-vs-minorized, notebooks/05_entrada_blanca.ipynb,
# commit 4e3102a).
M = 128
N = 96000
WARMUP = 500
AR_A = 0.0                  # white input
BETA = 0.2
VAR_THETA_0 = 2.0
FS = 8000
ROOM, T60, C_SOUND, SRC, MIC = [5, 10, 6], 0.2, 340, [1, 2.5, 2], [1, 1.5, 1]
ho = rir.generate(c=C_SOUND, fs=FS, r=MIC, s=SRC, L=ROOM, reverberation_time=T60, nsample=M).flatten()
ho = ho/np.linalg.norm(ho)
lags = np.abs(np.subtract.outer(np.arange(M), np.arange(M)))
Rxx = AR_A**lags
P_signal = ho @ Rxx @ ho                           # 1: white input, unit-norm response
w0 = np.zeros(M)


# === IGNACIO: laplace_scale and generate_signals - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized, notebooks/05_entrada_blanca.ipynb, commit 4e3102a
# generate_signals reads scale_gg off the globals; the sweeps below set it per SNR, as notebook 05 does.
def laplace_scale(var_eta):
    """b_eta = E|eta_t| of the generalized Gaussian of variance var_eta: the Laplacian closest to it
    in KL(p || q), the direction of eq. (10), and its maximum-likelihood fit."""
    scale = np.sqrt(var_eta/np.exp(gammaln(3/BETA) - gammaln(1/BETA)))   # generalized Gaussian scale
    return scale*np.exp(gammaln(2/BETA) - gammaln(1/BETA))               # E|eta| = scale Gamma(2/b)/Gamma(1/b)


def generate_signals(seed):
    """One realisation: the input, its clean output through ho, and the measurement noise."""
    rng = np.random.default_rng(seed)
    u = np.sqrt(1 - AR_A**2)*rng.standard_normal(N + WARMUP)   # driving noise, unit-variance x
    x = lfilter([1.0], [1.0, -AR_A], u)[WARMUP:]               # x_t = AR_A x_{t-1} + u_t
    y = np.convolve(ho, x)[:N]                                 # clean output, y_t = x_t' ho
    eta = gennorm.rvs(BETA, scale=scale_gg, size=N, random_state=rng)
    return x, y + eta, eta
# === end of the copied block ===


print(f"P_signal = {P_signal:.6f}")

## 3. The filters

Notebook 05's four robust filters, copied verbatim, for the check only. The sweep runs them in the
batched form of `fkf.ipynb`, with the closed-form corrections: the minorized one, eq. (45), and the exact
Laplacian one, eq. (43), which is the marginal filter of notebook 05 written from the joint posterior.

In [ ]:
# === IGNACIO: notebook 05's robust filters - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized, notebooks/05_entrada_blanca.ipynb, commit 4e3102a
def shift(new_x_sample, x_window):
    L = len(x_window)
    new_x_window = np.zeros(L)
    new_x_window[0] = new_x_sample
    new_x_window[1:] = x_window[:-1]
    return new_x_window


def sKF_L_minorized(n, x, d, w0, parameters):
    """Eqs. (robust.sKF.mean) and (robust.sKF.variance): the sKF above with v_eta -> b_eta |e_t|."""
    epsilon = parameters["epsilon"]                    # eps, process noise variance
    b_eta = parameters["b_eta"]                        # b_eta, Laplacian observation scale
    M = len(w0)                                        # number of weights
    w = w0.copy()                                      # w, current weights, start at w0
    v = parameters["var_theta_0"]                      # v, scalar prior variance on each weight
    x_t = np.zeros(M)                                  # x_t, window with the last M samples
    w_hist = np.zeros((n, M))                          # weights at every step
    e = np.zeros((n,))                                 # e_t, prediction error at every step
    sigma_hist = np.zeros((n,))                        # sigma_t at every step, only for k_t

    for k in range(n):
        x_t = shift(x[k], x_t)                         # new sample in, window slides
        e[k] = d[k] - x_t @ w                          # e_t = d_t - x_t' w_{t-1}

        w_hist[k] = w.copy()                           # state BEFORE the update, as Augusto does

        if k >= M:                                     # Augusto waits for a full window; same here
            v_tilde = v + epsilon                      # v_tilde = v + eps
            power = x_t @ x_t                          # ||x_t||^2
            denominator = b_eta*abs(e[k]) + v_tilde*power     # b_eta |e_t| + v_tilde ||x_t||^2
            w = w + x_t*(v_tilde*e[k]/denominator)     # w = w + v_tilde x_t e_t / denominator
            v = v_tilde*(1 - v_tilde*power/(M*denominator))   # v = v_tilde (1 - v_tilde ||x||^2/(M den))
            sigma_hist[k] = np.sqrt(v_tilde*power)     # sigma_t = sqrt(v_tilde ||x_t||^2), for k_t only

    return {"w_hist": w_hist, "e": e, "sigma": sigma_hist}


SIGN = np.array([1.0, -1.0])                           # the two branches, varsigma = +1 and -1


def softmax_two(log_weights):
    """Softmax of the two log weights: subtract the largest, exponentiate, normalise."""
    weights = np.exp(log_weights - log_weights.max())  # largest becomes 1, nothing overflows
    return weights/weights.sum()                       # sums to 1


def sKF_L_exact(n, x, d, w0, parameters):
    """Eqs. (exact.sKF.mean) and (exact.sKF.variance) of the draft: exact marginalization."""
    epsilon = parameters["epsilon"]                    # eps, process noise variance
    b_eta = parameters["b_eta"]                        # b_eta, Laplacian observation scale
    M = len(w0)                                        # number of weights
    w = w0.copy()                                      # w, current weights, start at w0
    v = parameters["var_theta_0"]                      # v, scalar prior variance on each weight
    x_t = np.zeros(M)                                  # x_t, window with the last M samples
    w_hist = np.zeros((n, M))                          # weights at every step
    e = np.zeros((n,))                                 # e_t, prediction error at every step
    sigma_hist = np.zeros((n,))                        # sigma_t at every step, only for k_t

    for k in range(n):
        x_t = shift(x[k], x_t)                         # new sample in, window slides
        e[k] = d[k] - x_t @ w                          # e_t = d_t - x_t' w_{t-1}

        w_hist[k] = w.copy()                           # state BEFORE the update, as Augusto does

        if k >= M:                                     # Augusto waits for a full window; same here
            v_tilde = v + epsilon                      # v_tilde = v + eps
            norm_x = np.sqrt(x_t @ x_t)                # ||x_t||
            power = norm_x**2                          # ||x_t||^2

            # The two mixture arguments. Global: no dependence on the weight index m.
            kappa = (SIGN*e[k] - v_tilde*power/b_eta)/(np.sqrt(v_tilde)*norm_x)

            # Mixture weights, in the log domain. Phi(kappa) underflows and e^{2 e / b_eta}
            # overflows, so neither factor may be formed on its own.
            log_Phi = log_ndtr(kappa)                  # log Phi(kappa), accurate in the left tail
            pi = softmax_two(-SIGN*e[k]/b_eta + log_Phi)

            # Inverse Mills ratio h = phi/Phi, also in the log domain.
            log_phi = -0.5*kappa**2 - 0.5*np.log(2*np.pi)
            h = np.exp(log_phi - log_Phi)

            Lambda = np.sum(SIGN*pi)                   # saturating prediction error, in [-1, 1]
            Gamma = np.sum(SIGN*pi*h)
            P = np.sum(pi*h)
            Q = np.sum(pi*kappa*h)

            w = w + (v_tilde*Lambda/b_eta - np.sqrt(v_tilde)*Gamma/norm_x)*x_t

            gamma = v_tilde/b_eta
            lam = np.sqrt(v_tilde)/norm_x
            D = (gamma**2*(1 - Lambda**2)
                 - 2*gamma*lam*(P - Lambda*Gamma)
                 - lam**2*(Q + Gamma**2))
            v = v_tilde + D*power/M                    # v = v_tilde + D ||x_t||^2 / M
            sigma_hist[k] = np.sqrt(v_tilde)*norm_x    # sigma_t = sqrt(v_tilde) ||x_t||, for k_t only

    return {"w_hist": w_hist, "e": e, "sigma": sigma_hist}


def fKF_L_minorized(n, x, d, w0, parameters):
    """Eq. (robust.fKF) of the draft: the Gaussian fKF above with v_eta -> b_eta |e_t|."""
    v = parameters["v"]                                # v, fixed variance on each weight
    b_eta = parameters["b_eta"]                        # b_eta, Laplacian observation scale
    M = len(w0)                                        # number of weights
    w = w0.copy()                                      # w, current weights, start at w0
    x_t = np.zeros(M)                                  # x_t, window with the last M samples
    w_hist = np.zeros((n, M))                          # weights at every step
    e = np.zeros((n,))                                 # e_t, prediction error at every step
    sigma_hist = np.zeros((n,))                        # sigma_t at every step, only for k_t
    gain_hist = np.zeros((n,))                         # gain on x_t e_t at every step

    for k in range(n):
        x_t = shift(x[k], x_t)                         # new sample in, window slides
        e[k] = d[k] - x_t @ w                          # e_t = d_t - x_t' w_{t-1}

        w_hist[k] = w.copy()                           # state BEFORE the update, as Augusto does

        if k >= M:                                     # Augusto waits for a full window; same here
            power = x_t @ x_t                          # ||x_t||^2
            gain = v/(b_eta*abs(e[k]) + v*power)       # g = v / (b_eta |e_t| + v ||x_t||^2)
            w = w + gain*x_t*e[k]                      # w = w + g x_t e_t
            gain_hist[k] = gain
            sigma_hist[k] = np.sqrt(v*power)           # sigma_t = sqrt(v ||x_t||^2), for k_t only

    return {"w_hist": w_hist, "e": e, "gain_hist": gain_hist, "sigma": sigma_hist}


def fKF_L_exact(n, x, d, w0, parameters):
    """Eq. (exact.fKF) of the draft: the sKF-L above with its variance fixed at v."""
    v = parameters["v"]                                # v, fixed variance on each weight
    b_eta = parameters["b_eta"]                        # b_eta, Laplacian observation scale
    M = len(w0)                                        # number of weights
    w = w0.copy()                                      # w, current weights, start at w0
    x_t = np.zeros(M)                                  # x_t, window with the last M samples
    w_hist = np.zeros((n, M))                          # weights at every step
    e = np.zeros((n,))                                 # e_t, prediction error at every step
    sigma_hist = np.zeros((n,))                        # sigma_t at every step, only for k_t
    gain_hist = np.zeros((n,))                         # gain on x_t e_t at every step

    for k in range(n):
        x_t = shift(x[k], x_t)                         # new sample in, window slides
        e[k] = d[k] - x_t @ w                          # e_t = d_t - x_t' w_{t-1}

        w_hist[k] = w.copy()                           # state BEFORE the update, as Augusto does

        if k >= M:                                     # Augusto waits for a full window; same here
            norm_x = np.sqrt(x_t @ x_t)                # ||x_t||
            power = norm_x**2                          # ||x_t||^2

            # The two mixture arguments. Global: no dependence on the weight index m.
            kappa = (SIGN*e[k] - v*power/b_eta)/(np.sqrt(v)*norm_x)

            # Mixture weights, in the log domain. Phi(kappa) underflows and e^{2 e / b_eta}
            # overflows, so neither factor may be formed on its own.
            log_Phi = log_ndtr(kappa)                  # log Phi(kappa), accurate in the left tail
            pi = softmax_two(-SIGN*e[k]/b_eta + log_Phi)

            # Inverse Mills ratio h = phi/Phi, also in the log domain.
            log_phi = -0.5*kappa**2 - 0.5*np.log(2*np.pi)
            h = np.exp(log_phi - log_Phi)

            Lambda = np.sum(SIGN*pi)                   # saturating prediction error, in [-1, 1]
            Gamma = np.sum(SIGN*pi*h)

            gain = v*Lambda/b_eta - np.sqrt(v)*Gamma/norm_x
            w = w + gain*x_t                           # w = w + (v L/b_eta - sqrt(v) G/||x||) x_t
            gain_hist[k] = gain/e[k]                   # the coefficient of x_t, over e_t
            sigma_hist[k] = np.sqrt(v)*norm_x          # sigma_t = sqrt(v) ||x_t||, for k_t only

    return {"w_hist": w_hist, "e": e, "gain_hist": gain_hist, "sigma": sigma_hist}
# === end of the copied block ===


# === IGNACIO: chi_laplacian - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized, notebooks/09_matcheado.ipynb, commit 6e6dabc
# (from notebook 07, f26ed36)
def log_mills(z):
    """log R(z), with R(z) = Phi(-z)/phi(z) the Mills ratio, eq. (40). R grows like e^{z^2/2} for
    negative z and overflows, so it is only ever handled through its logarithm."""
    return log_ndtr(-z) + 0.5*z**2 + 0.5*np.log(2*np.pi)


def chi_laplacian(e, sigma, b_eta):
    """Correction chi_t(e_t) and its slope chi'_t(e_t) for Laplacian noise, eqs. (39), (41) and (43)."""
    u = e/sigma                                        # u_t = e_t / sigma_t
    k_t = sigma/b_eta                                  # k_t = sigma_t / b_eta
    tau = sigma**2/b_eta                               # tau_t = sigma_t^2 / b_eta, the largest correction
    log_R_minus = log_mills(k_t - u)                   # log R(k_t - u_t)
    log_R_plus = log_mills(k_t + u)                    # log R(k_t + u_t)
    Lambda = np.tanh((log_R_minus - log_R_plus)/2)     # (R- - R+)/(R- + R+), in (-1, 1)
    chi = tau*Lambda                                   # chi = tau Lambda
    chi_slope = (2*k_t*np.exp(-np.logaddexp(log_R_minus, log_R_plus))   # 2k / (R- + R+)
                 - k_t**2*(1 - Lambda**2))                               # - k^2 (1 - Lambda^2)
    return chi, chi_slope
# === end of the copied block ===


# Copied from fkf.ipynb, commit 058a9f9.
def chi_minorized(e, sigma, b_eta):
    """Eq. (45) of the draft (chi.minorized): the minorized correction tau e/(tau + |e|), and the
    ratio chi/e that replaces chi' in the variance update (Table 2). The fKF uses the first only."""
    tau = sigma**2/b_eta                               # tau_t = sigma_t^2 / b_eta
    ratio = tau/(tau + np.abs(e))                      # chi_min / e, in (0, 1]
    return ratio*e, ratio


# sKF_batch and fKF_batch: copied from fkf.ipynb, commit 058a9f9 (sKF_batch from vkf-kf.ipynb,
# commit 2724d95), with options added and nothing else changed. Defaults reproduce the originals.
#   flip_at: misalignment against -h from that step on (fkf.ipynb had it for the fKF only)
#   clip:    chi' -> max(chi', 0) in the variance update, eq. (36); the mean update is untouched
#   record:  also return chi'_t and v_t at every step
#   v0:      initial v (VAR_THETA_0 = 2 in every notebook of ours; 1/M in Leszek's ggbench.py)
#   wait:    first step that updates (M: wait for a full window, as every notebook of ours)
def _roll_in(X_t, x_win):
    x_win = np.roll(x_win, 1, axis=1)
    x_win[:, 0] = X_t
    return x_win


def sKF_batch(X, D, h, b_eta, epsilon, chi_fn, flip_at=None, clip=False, record=False,
              v0=VAR_THETA_0, wait=None):
    L, (B, n) = len(h), X.shape
    first = L if wait is None else wait
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    v = np.full(B, float(v0))
    eps = np.asarray(epsilon, dtype=float)
    mis = np.empty((B, n))
    if record:
        slope_hist, v_hist = np.full((B, n), np.nan), np.full((B, n), np.nan)
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = (w - h) if flip_at is None or t < flip_at else (w + h)
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if t < first:
            continue
        v_tilde = v + eps
        power = np.einsum("bm,bm->b", x_t, x_t)
        sigma = np.sqrt(v_tilde*power)
        chi, slope = chi_fn(e, sigma, b_eta)
        w = w + x_t*(chi/power)[:, None]
        v = v_tilde*(1 - (np.maximum(slope, 0.0) if clip else slope)/L)
        if record:
            slope_hist[:, t], v_hist[:, t] = slope, v
    if record:
        return mis, slope_hist, v_hist
    return mis


def fKF_batch(X, D, h, b_eta, v, chi_fn, flip_at=None, wait=None):
    """B fKFs of eq. (37) side by side. Misalignment against h, or against -h from step flip_at on."""
    L, (B, n) = len(h), X.shape
    first = L if wait is None else wait
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    v = np.asarray(v, dtype=float)
    mis = np.empty((B, n))
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = (w - h) if flip_at is None or t < flip_at else (w + h)
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if t < first:
            continue
        power = np.einsum("bm,bm->b", x_t, x_t)
        chi, _ = chi_fn(e, np.sqrt(v*power), b_eta)
        w = w + x_t*(chi/power)[:, None]
    return mis


ROBUST = [("sKF-L (minorized)", "sKF", chi_minorized, sKF_L_minorized),
          ("sKF-L (exact)", "sKF", chi_laplacian, sKF_L_exact),
          ("fKF-L (minorized)", "fKF", chi_minorized, fKF_L_minorized),
          ("fKF-L (exact)", "fKF", chi_laplacian, fKF_L_exact)]


def run_batch(family, chi, X, D, values, b_eta):
    if family == "sKF":
        return sKF_batch(X, D, ho, b_eta, values, chi)
    return fKF_batch(X, D, ho, b_eta, values, chi)


# check: batched against notebook 05's scalar filters, realisation 0, 5 dB, 3000 steps
var_eta = P_signal/10**(5.0/10)
b_eta = laplace_scale(var_eta)
scale_gg = np.sqrt(var_eta/np.exp(gammaln(3/BETA) - gammaln(1/BETA)))
x_0, d_0, _ = generate_signals(0)
N_CMP = 3000
print(f"batched against notebook 05's scalar filters, 5 dB, realisation 0, {N_CMP} steps")
for name, family, chi, scalar_fn in ROBUST:
    value = 2e-6 if family == "sKF" else 4e-4
    parameters = {"b_eta": b_eta, "var_theta_0": VAR_THETA_0, ("epsilon" if family == "sKF" else "v"): value}
    mis_scalar = ((scalar_fn(N_CMP, x_0[:N_CMP], d_0[:N_CMP], w0, parameters)["w_hist"] - ho)**2).sum(axis=1)
    mis_batch = run_batch(family, chi, x_0[None, :N_CMP], d_0[None, :N_CMP], [value], b_eta)[0]
    print(f"   {name:<18} max relative difference = {np.max(np.abs(mis_batch - mis_scalar)/mis_scalar):.1e}")

## 4. The search, and a control at notebook 05's fixed target

Notebook 05's protocol, copied: at each SNR, scan the grid at $R = 3$ over $N = 48000$ steps, keep the
settled points, interpolate the value that lands on the target, and run it at $R = 10$. A target outside
the settled floors gives no point. Grids: logspace$(-11, -3, 17)$ for $\varepsilon$, logspace$(-6, 0, 13)$ for $v$.

**Control:** at the fixed $-20$ dB target this must repeat notebook 05's table (its section 6).

In [ ]:
# === IGNACIO: steady_state and is_settled - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized, notebooks/05_entrada_blanca.ipynb, commit 4e3102a
def steady_state(misalignment):
    """Floor in dB, and the first step within 3 dB of it."""
    tail = slice(3*len(misalignment)//4, len(misalignment))
    floor = 10*np.log10(misalignment[tail].mean())
    db = 10*np.log10(misalignment)
    reached = int(np.argmax(db < floor + 3))
    return floor, reached


def is_settled(misalignment):
    """Settled within the run: within 3 dB of the floor by half of it, and no longer descending."""
    n = len(misalignment)
    floor, reached = steady_state(misalignment)
    third_quarter = 10*np.log10(misalignment[n//2:3*n//4].mean())
    # reached = 0: the run never left the 3 dB band around its start, so it never converged either
    return bool(0 < reached <= n/2 and third_quarter - floor <= DRIFT_DB)
# === end of the copied block ===



DRIFT_DB = 0.5
R_SEARCH = 3
R_PLOT = 10
N_SWEEP = 48000
EPS_GRID_SNR = np.logspace(-11, -3, 17)
V_GRID_SNR = np.logspace(-6, 0, 13)
GRID = {"sKF": EPS_GRID_SNR, "fKF": V_GRID_SNR}


def search_at_snr(family, chi, grid, b_eta, search, plot, target):
    """search_at_snr of notebook 05 with the grid run in one batch and the target as a parameter."""
    g = len(grid)
    X, D = search
    curves = run_batch(family, chi, np.repeat(X, g, axis=0), np.repeat(D, g, axis=0),
                       np.tile(grid, len(X)), b_eta).reshape(len(X), g, -1).mean(axis=0)
    floors = np.array([steady_state(c)[0] for c in curves])
    settled = np.array([is_settled(c) for c in curves])
    if not settled.any():
        return {"value": np.nan, "floor": np.nan, "steps": -1, "best": np.nan}
    values, kept = grid[settled], floors[settled]
    if target < kept.min() or target > kept.max():
        return {"value": np.nan, "floor": np.nan, "steps": -1, "best": kept.min()}
    order = np.argsort(kept)
    value = 10**np.interp(target, kept[order], np.log10(values)[order])
    floor, reached = steady_state(run_batch(family, chi, plot[0], plot[1], np.full(len(plot[0]), value),
                                            b_eta).mean(axis=0))
    return {"value": value, "floor": floor, "steps": reached, "best": kept.min()}


def snr_sweep(snrs, target_of, grids=GRID, filters=ROBUST):
    global var_eta, b_eta, scale_gg                    # read by generate_signals, as in notebook 05
    table = {}
    for snr in snrs:
        var_eta = P_signal/10**(snr/10)
        b_eta = laplace_scale(var_eta)
        scale_gg = np.sqrt(var_eta/np.exp(gammaln(3/BETA) - gammaln(1/BETA)))
        sigs = [generate_signals(seed) for seed in range(R_PLOT)]      # seeds 0-2 search, 0-9 plot
        search = (np.array([s[0][:N_SWEEP] for s in sigs[:R_SEARCH]]), np.array([s[1][:N_SWEEP] for s in sigs[:R_SEARCH]]))
        plot = (np.array([s[0][:N_SWEEP] for s in sigs]), np.array([s[1][:N_SWEEP] for s in sigs]))
        for name, family, chi, _ in filters:
            table[(snr, name)] = search_at_snr(family, chi, grids[family], b_eta, search, plot, target_of(snr))
    return table


def print_table(table, snrs, target_of, filters=ROBUST):
    print(f"{'SNR':>5}{'target':>8}  {'filter':<20}{'parameter':>11}{'floor [dB]':>12}{'steps':>8}")
    for snr in snrs:
        for name, *_ in filters:
            r = table[(snr, name)]
            value = "-" if np.isnan(r["value"]) else f"{r['value']:.2e}"
            note = "" if r["steps"] > 0 else f"   no point: deepest settled floor {r['best']:+.2f} dB"
            print(f"{snr:>5.0f}{target_of(snr):>8.0f}  {name:<20}{value:>11}{r['floor']:>12.2f}{r['steps']:>8d}{note}")


def ratio(table, snr, family):
    a, b = table[(snr, f"{family}-L (exact)")]["steps"], table[(snr, f"{family}-L (minorized)")]["steps"]
    return a/b if a > 0 and b > 0 else np.nan


FIXED_SNR = [0.0, 5.0, 10.0, 15.0]
fixed_target = lambda snr: -20.0
start = time.time()
fixed = snr_sweep(FIXED_SNR, fixed_target)
print(f"fixed target done in {time.time() - start:.0f} s\n")
print_table(fixed, FIXED_SNR, fixed_target)

# notebook 05, section 6, printed steps: sKF-L min, sKF-L exact, fKF-L min, fKF-L exact
NB05 = {0.0: (1676, 1778, 1419, 1431), 5.0: (1508, 1571, 1066, 1068),
        10.0: (1274, 1502, 804, 802), 15.0: (1056, 1205, 652, 636)}
same = all(tuple(fixed[(snr, name)]["steps"] for name, *_ in ROBUST) == steps for snr, steps in NB05.items())
print(f"\ncontrol: all 16 step counts equal to notebook 05's: {same}")

## 5. The moving target, $-(\mathrm{SNR} + 15)$ dB

In [ ]:
SWEEP_SNR = [0.0, 5.0, 10.0, 15.0, 20.0, 25.0, 30.0]
moving_target = lambda snr: -(snr + 15.0)
start = time.time()
moving = snr_sweep(SWEEP_SNR, moving_target)
print(f"moving target done in {time.time() - start:.0f} s\n")
print_table(moving, SWEEP_SNR, moving_target)

At 30 dB the exact fKF-L has no point: its deepest settled floor is $-44.81$ dB, just above the $-45$ dB
target. Is that the grid, which stops at $v = 10^{-6}$? As a check, outside notebook 05's protocol, the
fKF-L pair at 30 dB is run again with the grid extended one decade down, logspace$(-7, 0, 15)$.

In [ ]:
FKF_ONLY = [f for f in ROBUST if f[1] == "fKF"]
extended = snr_sweep([30.0], moving_target, grids={"fKF": np.logspace(-7, 0, 15)}, filters=FKF_ONLY)
print_table(extended, [30.0], moving_target, filters=FKF_ONLY)

**Figure 1.** Steps to converge, exact over minorized, at equal floor. Solid: the moving target
$-(\mathrm{SNR} + 15)$ dB. Dashed: notebook 05's fixed $-20$ dB (the control above). Grey, dotted: the sKF-L
under AR($-0.9$) input with the moving target, from `04_escenario_realista_large_k_mad.ipynb`, for
comparison. The fKF-L has no point at 30 dB (see above). Above 1, the minorized filter is faster.

In [ ]:
COLOUR = {"sKF": "#2a78d6", "fKF": "#eb6834"}      # one colour per family (palette of fkf.ipynb)
MARKER = {"sKF": "o", "fKF": "s"}
NB04_AR = [1.11, 1.09, 1.06, 1.01, 0.99, 0.96, 0.93]   # DECISIONS.md, 2026-09-21

fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)
for family in ("sKF", "fKF"):
    ax.plot(SWEEP_SNR, [ratio(moving, s, family) for s in SWEEP_SNR], marker=MARKER[family],
            color=COLOUR[family], lw=1.6, label=f"{family}-L, white input, target -(SNR + 15) dB")
    ax.plot(FIXED_SNR, [ratio(fixed, s, family) for s in FIXED_SNR], marker=MARKER[family], mfc="none",
            color=COLOUR[family], ls="--", lw=1.1, label=f"{family}-L, white input, target -20 dB (notebook 05)")
ax.plot(SWEEP_SNR, NB04_AR, color="0.5", ls=":", marker=".", lw=1.2,
        label="sKF-L, AR(-0.9), target -(SNR + 15) dB (notebook 04, mad)")
ax.axhline(1.0, color="k", lw=0.8)
ax.set(xlabel="SNR [dB]", ylabel="steps to converge, exact / minorized", xticks=SWEEP_SNR,
       title="Speed at equal floor: above 1, the minorized filter is faster")
ax.grid(alpha=0.25)
ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False)
plt.show()

print(f"{'SNR':>5}{'target':>8}{'sKF-L':>8}{'fKF-L':>8}{'sKF-L, -20 dB':>15}{'fKF-L, -20 dB':>15}{'sKF-L, AR':>11}")
for i, snr in enumerate(SWEEP_SNR):
    f_moving = ratio(moving, snr, "fKF")
    fixed_s = f"{ratio(fixed, snr, 'sKF'):.2f}" if snr in FIXED_SNR else "-"
    fixed_f = f"{ratio(fixed, snr, 'fKF'):.2f}" if snr in FIXED_SNR else "-"
    print(f"{snr:>5.0f}{moving_target(snr):>8.0f}{ratio(moving, snr, 'sKF'):>8.2f}{f_moving:>8.2f}"
          f"{fixed_s:>15}{fixed_f:>15}{NB04_AR[i]:>11.2f}")
print("   exact / minorized steps; nan: no point, the target is outside the settled floors")

## 6. Findings

**Checks.** The batched filters equal notebook 05's scalar ones, and at the fixed $-20$ dB target all 16 step
counts of notebook 05's SNR table come out the same. So the protocol was copied correctly, and only the
target differs below.

**Notebook 07 needs no new run.** Its scenario and filter pair are those of
`04_escenario_realista_large_k_mad.ipynb`, which already has the moving target at $\mathrm{E}|\eta|$.

**White input, moving target (exact / minorized):**

| SNR [dB] | 0 | 5 | 10 | 15 | 20 | 25 | 30 |
|---|---|---|---|---|---|---|---|
| target [dB] | $-15$ | $-20$ | $-25$ | $-30$ | $-35$ | $-40$ | $-45$ |
| sKF-L | 1.11 | 1.04 | 1.06 | 1.04 | 1.09 | 1.04 | 1.05 |
| fKF-L | 0.98 | 1.00 | 1.03 | 1.10 | 1.17 | 1.22 | - |
| sKF-L, AR($-0.9$), notebook 04 | 1.11 | 1.09 | 1.06 | 1.01 | 0.99 | 0.96 | 0.93 |

- **sKF-L:** the exact filter is 4 to 11 % slower at every SNR, with no trend. Notebook 05's fixed target gave
  1.06, 1.04, 1.18, 1.14 at 0-15 dB; with the moving target the 10 and 15 dB points drop to 1.06 and 1.04.
  Under AR($-0.9$) input (notebook 04) the ratio falls with SNR and crosses 1 near 20 dB; under white input
  it does not.
- **fKF-L:** equal at low SNR, and the exact filter falls behind as the SNR rises: 1.10 at 15 dB, 1.22 at
  25 dB. The fixed target hid this: notebook 05 gave 1.01, 1.00, 1.00, 0.98 at 0-15 dB, because at 15 dB a
  $-20$ dB target sits only 5 dB below the common NLMS floor, where the two filters coincide.

- **30 dB, fKF-L:** no point. The exact filter's deepest settled floor is $-44.81$ dB, 0.2 dB short of the
  target, and extending the grid one decade down gives no deeper settled point. So the limit is the
  48000-step runs of notebook 05's protocol, not the grid. The minorized fKF-L needs 8612 steps there.

So the caveat printed under notebook 05 was right: with white input, the fixed target made the fKF-L look
flat across SNR, and the moving target shows the exact fKF-L slowing down at high SNR.

In [ ]:
print(f"whole notebook: {(time.time() - NOTEBOOK_START)/60:.1f} min")